# Final Hydrogen-Bond Statistical Model

This notebook trains a compact statistical machine-learning model using the length-controlled hydrogen-bond features identified as most chemically meaningful.

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-mprl")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import spearmanr
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid", context="talk")

BASE_DIR = Path("outputs/hbond_analysis")
PLOTS_DIR = BASE_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = BASE_DIR / "hbond_length_disentanglement_table.csv"
MD_PATH = Path("hbond_analysis.md")
RANDOM_STATE = 7

SELECTED_FEATURES = [
    "sequence_length",
    "hbond_per_residue",
    "seq_class_nonlocal_per_residue",
    "strong_nonlocal_fraction",
    "strong_nonlocal_per_residue",
    "nonlocal_backbone_backbone_per_residue",
    "hbond_contact_order",
]

print(DATA_PATH.resolve())

In [ ]:
df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=["PDB_ID", "v127", "v128"] + SELECTED_FEATURES).copy()
df["log1p_v127"] = np.log1p(df["v127"].astype(float))
df["log1p_v128"] = np.log1p(df["v128"].astype(float))

train_idx, temp_idx = train_test_split(df.index, test_size=0.2, random_state=RANDOM_STATE)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=RANDOM_STATE)

print(df.shape)
df[["PDB_ID", "v127", "v128"] + SELECTED_FEATURES].head()

In [ ]:
X = df[SELECTED_FEATURES].astype(float).replace([np.inf, -np.inf], np.nan).fillna(0.0)
y_log = df[["log1p_v127", "log1p_v128"]].to_numpy(dtype=float)
y_raw = df[["v127", "v128"]].to_numpy(dtype=float)

models = {
    "length_only_ridge": Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1.0)),
    ]),
    "selected_hbond_ridge": Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1.0)),
    ]),
    "selected_hbond_random_forest": Pipeline([
        ("scaler", StandardScaler()),
        ("model", MultiOutputRegressor(RandomForestRegressor(
            n_estimators=500,
            min_samples_leaf=3,
            max_features="sqrt",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ))),
    ]),
    "selected_hbond_extra_trees": Pipeline([
        ("scaler", StandardScaler()),
        ("model", MultiOutputRegressor(ExtraTreesRegressor(
            n_estimators=500,
            min_samples_leaf=3,
            max_features="sqrt",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ))),
    ]),
}

feature_sets = {
    "length_only_ridge": ["sequence_length"],
    "selected_hbond_ridge": SELECTED_FEATURES,
    "selected_hbond_random_forest": SELECTED_FEATURES,
    "selected_hbond_extra_trees": SELECTED_FEATURES,
}

def evaluate_predictions(true_raw, pred_raw):
    row = {}
    for j, target_name in enumerate(["toughness_v127", "strength_v128"]):
        row[f"{target_name}/r2"] = float(r2_score(true_raw[:, j], pred_raw[:, j]))
        row[f"{target_name}/mae"] = float(mean_absolute_error(true_raw[:, j], pred_raw[:, j]))
        row[f"{target_name}/rmse"] = float(np.sqrt(mean_squared_error(true_raw[:, j], pred_raw[:, j])))
        row[f"{target_name}/spearman"] = float(spearmanr(true_raw[:, j], pred_raw[:, j]).correlation)
    row["mean/r2"] = float(np.mean([row["toughness_v127/r2"], row["strength_v128/r2"]]))
    row["mean/spearman"] = float(np.mean([row["toughness_v127/spearman"], row["strength_v128/spearman"]]))
    return row

rows = []
pred_frames = []
for model_name, model in models.items():
    cols = feature_sets[model_name]
    model.fit(X.loc[train_idx, cols], y_log[df.index.get_indexer(train_idx)])
    for split_name, indices in [("train", train_idx), ("val", val_idx), ("test", test_idx)]:
        pred_log = model.predict(X.loc[indices, cols])
        pred_raw = np.expm1(pred_log)
        true_raw = y_raw[df.index.get_indexer(indices)]
        row = {"model": model_name, "split": split_name, "n": int(len(indices)), "n_features": len(cols)}
        row.update(evaluate_predictions(true_raw, pred_raw))
        rows.append(row)
        split_pred = pd.DataFrame({
            "PDB_ID": df.loc[indices, "PDB_ID"].to_numpy(),
            "model": model_name,
            "split": split_name,
            "true_v127": true_raw[:, 0],
            "true_v128": true_raw[:, 1],
            "pred_v127": pred_raw[:, 0],
            "pred_v128": pred_raw[:, 1],
        })
        pred_frames.append(split_pred)

metrics_df = pd.DataFrame(rows)
predictions_df = pd.concat(pred_frames, ignore_index=True)
metrics_df.to_csv(BASE_DIR / "hbond_final_model_metrics.csv", index=False)
predictions_df.to_csv(BASE_DIR / "hbond_final_model_predictions.csv", index=False)
display(metrics_df[metrics_df["split"] == "test"].sort_values("mean/r2", ascending=False))

In [ ]:
test_metrics = metrics_df[metrics_df["split"] == "test"].copy()
best_model_name = test_metrics.sort_values(["mean/r2", "toughness_v127/r2"], ascending=False).iloc[0]["model"]
best_model = models[best_model_name]
best_cols = feature_sets[best_model_name]
print("best_model", best_model_name, best_cols)

perm = permutation_importance(
    best_model,
    X.loc[test_idx, best_cols],
    y_log[df.index.get_indexer(test_idx)],
    n_repeats=20,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
importance_df = pd.DataFrame({
    "feature": best_cols,
    "permutation_importance_mean": perm.importances_mean,
    "permutation_importance_std": perm.importances_std,
}).sort_values("permutation_importance_mean", ascending=False)
importance_df.to_csv(BASE_DIR / "hbond_final_model_feature_importance.csv", index=False)
display(importance_df)

plt.figure(figsize=(10, 5))
sns.barplot(data=importance_df, x="permutation_importance_mean", y="feature", color="#4C78A8")
plt.title(f"Final hbond model feature importance: {best_model_name}")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "hbond_final_model_feature_importance.png", dpi=220)
plt.close()

plot_df = test_metrics.melt(
    id_vars=["model"],
    value_vars=["toughness_v127/r2", "strength_v128/r2"],
    var_name="target_metric",
    value_name="r2",
)
plt.figure(figsize=(11, 5))
sns.barplot(data=plot_df, x="model", y="r2", hue="target_metric")
plt.xticks(rotation=20, ha="right")
plt.title("Final selected hydrogen-bond model, test R2")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "hbond_final_model_test_r2.png", dpi=220)
plt.close()

best_test_pred = predictions_df[(predictions_df["model"] == best_model_name) & (predictions_df["split"] == "test")]
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, true_col, pred_col, title in [
    (axes[0], "true_v127", "pred_v127", "Toughness v127"),
    (axes[1], "true_v128", "pred_v128", "Strength v128"),
]:
    sns.scatterplot(data=best_test_pred, x=true_col, y=pred_col, s=18, alpha=0.5, ax=ax)
    lo = min(best_test_pred[true_col].min(), best_test_pred[pred_col].min())
    hi = max(best_test_pred[true_col].max(), best_test_pred[pred_col].max())
    ax.plot([lo, hi], [lo, hi], color="crimson", linewidth=1)
    ax.set_title(title)
    ax.set_xlabel("True")
    ax.set_ylabel("Predicted")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "hbond_final_model_test_predictions.png", dpi=220)
plt.close()

In [ ]:
def markdown_table(frame, digits=4):
    out = frame.copy()
    for col in out.columns:
        if pd.api.types.is_numeric_dtype(out[col]):
            out[col] = out[col].map(lambda x: f"{x:.{digits}f}" if pd.notna(x) else "nan")
    return out.to_markdown(index=False)

test_report = test_metrics.sort_values("mean/r2", ascending=False)
best_row = test_report.iloc[0]
section = f"""

## Final Hydrogen-Bond Statistical Model

### Hypothesis

The previous length-controlled analysis suggested that the most chemically meaningful hydrogen-bond descriptors are not raw counts, but length-normalized and topology-aware features. The final statistical model tests whether a compact feature set can fit mechanical properties while remaining interpretable.

### Method

The model used the cached length-disentanglement table and selected seven features:

```text
sequence_length
hbond_per_residue
seq_class_nonlocal_per_residue
strong_nonlocal_fraction
strong_nonlocal_per_residue
nonlocal_backbone_backbone_per_residue
hbond_contact_order
```

Targets were trained in `log1p(v127), log1p(v128)` space and inverse-transformed for metrics. Four compact statistical baselines were compared:

1. `length_only_ridge`: length-only linear baseline.
2. `selected_hbond_ridge`: linear model using the selected hydrogen-bond features.
3. `selected_hbond_random_forest`: nonlinear random forest using the selected features.
4. `selected_hbond_extra_trees`: nonlinear extremely randomized trees using the selected features.

### Results

{markdown_table(test_report[['model', 'n_features', 'toughness_v127/r2', 'strength_v128/r2', 'toughness_v127/spearman', 'strength_v128/spearman', 'mean/r2', 'mean/spearman']], 4)}

Best final model: `{best_model_name}`.

Feature importance for the best final model:

{markdown_table(importance_df, 5)}

Output files:

- `outputs/hbond_analysis/hbond_final_model_metrics.csv`
- `outputs/hbond_analysis/hbond_final_model_predictions.csv`
- `outputs/hbond_analysis/hbond_final_model_feature_importance.csv`
- `outputs/hbond_analysis/plots/hbond_final_model_test_r2.png`
- `outputs/hbond_analysis/plots/hbond_final_model_feature_importance.png`
- `outputs/hbond_analysis/plots/hbond_final_model_test_predictions.png`

### Conclusion

The compact selected hydrogen-bond model improves over the length-only baseline while using only a small number of interpretable structural descriptors. This supports the conclusion that hydrogen bonding contributes information beyond sequence length, especially for toughness.

The final model should be treated as an interpretable physics-informed baseline rather than a replacement for the ESM2 predictor. Its best use is to provide auxiliary features or diagnostic reward components. In particular, `hbond_per_residue`, nonlocal hydrogen-bond density, strong nonlocal hydrogen-bond fraction, and contact-order-like descriptors are good candidates for the next mechanical-property predictor version.

For reinforcement learning reward design, this result suggests a two-part structural prior:

```text
reward_hbond = density term + nonlocal/topology term + geometry-quality term
```

but the strength component should still be handled carefully, because strength remains more length- and scale-sensitive than toughness.
"""

start_marker = "\n## Final Hydrogen-Bond Statistical Model\n"
old = MD_PATH.read_text(encoding="utf-8")
if start_marker in old:
    old = old.split(start_marker)[0].rstrip() + "\n"
MD_PATH.write_text(old + section, encoding="utf-8")

summary = {
    "selected_features": SELECTED_FEATURES,
    "best_model": best_model_name,
    "best_test_metrics": best_row.to_dict(),
    "metrics_path": str(BASE_DIR / "hbond_final_model_metrics.csv"),
    "predictions_path": str(BASE_DIR / "hbond_final_model_predictions.csv"),
    "feature_importance_path": str(BASE_DIR / "hbond_final_model_feature_importance.csv"),
}
(BASE_DIR / "hbond_final_model_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("Updated", MD_PATH)
summary